# Blocco 3 ? Scoperta dell'Object-Centric Petri Net

## Obiettivo

Scoprire un Object-Centric Petri Net (OCPN) a partire dal dataset
Order Management in formato OCEL 2.0.

A differenza di una Petri net tradizionale basata su un unico case
identifier, una OCPN mantiene distinti i diversi tipi di oggetto
coinvolti nello stesso processo.

## 1. Caricamento del dataset e configurazione

Il dataset viene caricato tramite la funzione centralizzata
`load_ocel2_sqlite`. I percorsi degli artefatti sono ricavati dalla
configurazione del progetto, cos? il notebook pu? essere eseguito sia
dalla root sia dalla cartella `notebooks` senza generare output in
posizioni errate.

In [ ]:
import pm4py

from ocpm_partial_order.config import (
    FIGURES_DIR,
    MAIN_DATASET_DB,
    REPORTS_DIR,
)
from ocpm_partial_order.discovery.ocpn_discovery import (
    discover_ocpn,
)
from ocpm_partial_order.io.ocel_loader import (
    load_ocel2_sqlite,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

ocel = load_ocel2_sqlite(MAIN_DATASET_DB)




  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




## 2. Scoperta del modello

La funzione `discover_ocpn` applica l'algoritmo di discovery
object-centric fornito da PM4Py. Il risultato ? un dizionario che
contiene il modello e le strutture statistiche utilizzate per
costruirlo.

In [ ]:
ocpn = discover_ocpn(ocel)

print("Tipo risultato:", type(ocpn))
print("Chiavi principali:")
print(ocpn.keys())

## 3. Visualizzazione interattiva

La cella seguente apre la rappresentazione grafica della OCPN.
Richiede Graphviz installato e disponibile nel `PATH` di sistema.

In [ ]:
pm4py.view_ocpn(ocpn)

## 4. Esportazione della figura

La visualizzazione viene salvata in `outputs/figures`. L'immagine ?
un artefatto rigenerabile e non viene versionata da Git.

In [ ]:
ocpn_output_path = (
    FIGURES_DIR / "order_management_ocpn.png"
)

pm4py.save_vis_ocpn(
    ocpn,
    str(ocpn_output_path),
)

print("Modello salvato in:", ocpn_output_path)

## 5. Ispezione della struttura restituita

L'ispezione permette di verificare quali componenti siano state
prodotte dalla discovery senza assumere che sia gi? stato effettuato
il conformance checking.

In [ ]:
for key, value in ocpn.items():
    print("=" * 70)
    print("CHIAVE:", key)
    print("TIPO:", type(value))

    if isinstance(value, dict):
        print("SOTTOCHIAVI:", list(value.keys())[:20])
    elif isinstance(value, (list, set, tuple)):
        print("DIMENSIONE:", len(value))
        print("PRIMI ELEMENTI:", list(value)[:5])
    else:
        print("VALORE:", value)

## 6. Esportazione del report strutturale

La struttura principale viene salvata in `outputs/reports`. Anche
questo file ? un artefatto generato automaticamente e viene escluso
dal repository.

In [ ]:
summary_path = (
    REPORTS_DIR / "order_management_ocpn_structure.txt"
)

with summary_path.open(
    "w",
    encoding="utf-8",
) as file:
    for key, value in ocpn.items():
        file.write(f"KEY: {key}\n")
        file.write(f"TYPE: {type(value)}\n")

        if isinstance(value, dict):
            file.write(
                f"SUBKEYS: {list(value.keys())}\n"
            )
        elif isinstance(value, (list, set, tuple)):
            file.write(f"SIZE: {len(value)}\n")
        else:
            file.write(f"VALUE: {value}\n")

        file.write("\n")

print("Report salvato in:", summary_path)


## Conclusioni

La discovery preliminare produce una OCPN relativa a 11 attivit? e
5 tipi di oggetto: `orders`, `items`, `packages`, `products` ed
`employees`.

Il risultato comprende una Petri net per ciascun tipo di oggetto,
oltre alle informazioni su archi, attivit? iniziali e finali,
molteplicit? e prestazioni.

Questo blocco completa la scoperta e l'ispezione strutturale del
modello. Non dimostra ancora che una specifica process execution sia
fitting e non calcola una fitness pari a 1. La verifica di conformit?
dovr? essere realizzata in un blocco successivo.